# L3b: Data Validation and Provenance
In this lab, we read reactor-run records from CSV and their metadata from JSON. We will inspect provenance before values, validate the data contract, diagnose an intentionally invalid data example, and verify byte-level integrity.

> __Learning Objectives__
>
> * __Load a multi-file data bundle:__ Read tabular observations and structured metadata into appropriate Julia representations.
> * __Interrogate provenance:__ Determine what the dataset is and what claims it cannot support.
> * __Diagnose validation failures:__ Connect error messages to physical and structural constraints.
> * __Compare language implementations:__ Recognize the same data contract in Julia and Python.

Let's get started!
___

## Setup, Data, and Prerequisites
We use the root CHEME 5800 Julia environment through the local `Include.jl`. The class-meeting folder contains the observation file, metadata, invalid data example, checksum record, and source code, so the notebook requires no network access.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the local setup file in the notebook's global scope. This local file delegates environment activation to the repository root and loads the L3b source module.

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # load the shared environment and L3b source

### Data
The CSV observation table contains eight instructor-generated synthetic reactor runs. CSV represents repeated records as rows with common named columns. Each row records a run identifier, catalyst label, temperature, residence time, and conversion fraction. The JSON file represents structured metadata: source, intended purpose, units, expected row count, and validation constraints.

> __Parsing versus validation__: A parser can determine whether bytes follow the CSV or JSON syntax. It does not know that residence time must be positive, conversion must lie between zero and one, or run identifiers must be unique. Those requirements belong to the data contract.

> __Important__: These values are synthetic teaching data. They are useful for exercising a data pipeline but are not measurements from a physical reactor.

In [ ]:
data_root = joinpath(CHEME5800_L3B_ROOT, "data");
csv_path = joinpath(data_root, "reactor-runs.csv");
metadata_path = joinpath(data_root, "reactor-metadata.json");
invalid_path = joinpath(data_root, "reactor-runs-invalid-example.csv");

___

## Task 1: Inspect Provenance Before Observations
We begin by loading the complete bundle, but we inspect the metadata before drawing attention to the numerical table. The provenance field determines whether the values may be interpreted as measurements, simulations, or teaching examples.

In [ ]:
bundle = L3bData.load_reactor_bundle(csv_path, metadata_path);
provenance_record = (
    dataset_id = bundle.metadata["dataset_id"],
    provenance = bundle.metadata["provenance"],
    purpose = bundle.metadata["purpose"],
    units = bundle.metadata["units"],
)

The record identifies the creator and purpose explicitly: these are instructor-generated synthetic values for practicing a computational workflow. We may use them to test code and reason about file structure, but not to estimate a physical kinetic law or compare real catalysts.

___

## Task 2: Validate the Authored Observation Table
The loader parses the files and then checks required columns, nonempty identifiers, identifier uniqueness, finite numerical values, physical domains, and agreement between the metadata row count and CSV row count.

In [ ]:
bundle.validation

The empty error list means that the authored files satisfy the declared computational contract. Now—and only now—we inspect the table. Passing validation does not change its synthetic provenance.

In [ ]:
bundle.runs

___

## Task 3: Diagnose a Known-Bad Data Example
The second CSV is intentionally invalid. One row reports zero residence time and a conversion fraction greater than one. We keep this file because repeatable failure cases are necessary to test a validator.

In [ ]:
invalid_runs = CSV.read(invalid_path, DataFrame);
invalid_report = L3bData.validate_reactor_runs(invalid_runs)

The report returns both physical-domain failures in one pass. This is more useful to a data owner than stopping at the first problem, because both values can be repaired before validation is run again.

___

## Task 4: Verify Integrity and Compare the Python Contract
We compute the CSV SHA-256 digest and compare it with the digest recorded in `data/README.md`. Matching values establish that our file has the authored bytes; they do not establish that the synthetic values are experimental truth.

In [ ]:
csv_sha256 = L3bData.file_sha256(csv_path);
authored_sha256 = "018dc9662e309456da77decc89274075385349cd25fd07e71b6ca789d86fdee3";
(sha256 = csv_sha256, matches_authored_file = csv_sha256 == authored_sha256)

The file `src/reactor_data.py` expresses the same record and metadata checks using only the Python standard library. The syntax and runtime types differ, but the contract does not. From the repository root, run:

```bash
python weeks/week-03/L3b/src/reactor_data.py
python -m unittest discover -s weeks/week-03/L3b/src -p 'test_*.py'
```

In [ ]:
@testset "L3b data contract" begin
    @test bundle.validation.valid
    @test nrow(bundle.runs) == bundle.metadata["expected_rows"]
    @test occursin("synthetic", lowercase(bundle.metadata["provenance"]))
    @test !invalid_report.valid
    @test any(error -> occursin("residence_time_min", error), invalid_report.errors)
    @test any(error -> occursin("conversion_fraction", error), invalid_report.errors)
    @test csv_sha256 == authored_sha256
end

## Summary
We treated the reactor records and their metadata as one data bundle, inspected origin before values, validated the declared contract, retained a known-bad example, and verified file integrity.

> __Key Takeaways__
>
> * __Provenance comes first:__ It determines which interpretations are responsible.
> * __Validation is executable domain knowledge:__ Units, ranges, uniqueness, and row counts become checks.
> * __Known-bad examples have instructional value:__ They make failure behavior reproducible.
> * __Contracts cross languages:__ Julia and Python can enforce the same requirements with different syntax.

___